# Thai Doc Classifier — run the 7B on a free Kaggle GPU

Runs `bench_local.py` with **Qwen2.5-VL-7B** on Kaggle's free **T4 (16 GB)** GPU — the model your 6 GB laptop can't host. Kaggle gives ~30 GPU-hrs/week free.

## One-time setup (do this in the right sidebar BEFORE running)
1. **Add your code as a Dataset:** zip the project folder (with `thaidoc_llm/`, `thaidoc/`, `bench_local.py`, and your `test-files/` inside) → **+ Add Input → Datasets → Upload → New Dataset** → attach it. Kaggle unzips it under `/kaggle/input/<name>/`.
2. **Accelerator → GPU T4 x2** (or P100).
3. **Internet → On** (required to pip-install and download the model).

## 0. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('CUDA available:', torch.cuda.is_available(), '| torch', torch.__version__)

## 1. Get the code (from the attached Dataset)
`/kaggle/input` is read-only, so we copy the project into the writable `/kaggle/working` and `cd` there.

In [ ]:
import os, glob, shutil

# auto-find the folder under /kaggle/input that contains thaidoc_llm/
src = next(os.path.dirname(p)
           for p in glob.glob('/kaggle/input/**/thaidoc_llm', recursive=True))
dst = '/kaggle/working/project'
if not os.path.exists(dst):
    shutil.copytree(src, dst)
os.chdir(dst)
print('Working dir:', os.getcwd())
print('Has bench_local.py:', os.path.exists('bench_local.py'))

## 2. Install dependencies
Kaggle ships CUDA PyTorch — we only add the model libraries. (Requires **Internet → On**.)

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes qwen-vl-utils \
    openpyxl pillow opencv-python-headless scikit-learn matplotlib
print('deps installed')

## 3. Point at your test images
Uses the `test-files/` folder from your dataset — the normal bench input. Make sure it was included in the zip.

In [ ]:
import os

DATA_DIR = 'test-files'   # the real test images from your dataset
exts = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp', '.webp')
imgs = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(exts)]
print('images:', len(imgs), 'in', DATA_DIR)
for f in sorted(imgs):
    print(' -', f)

## 4. Run the benchmark on the 7B
First run downloads the 7B weights (~16 GB) into the session. On the 16 GB T4 the 7B loads fully on-GPU in 4-bit (no CPU offload), ~2–4 s/image.

In [ ]:
!python bench_local.py --provider transformers \
    --model Qwen/Qwen2.5-VL-7B-Instruct --dir $DATA_DIR

### Compare against the 3B (same data)

In [ ]:
!python bench_local.py --provider transformers \
    --model Qwen/Qwen2.5-VL-3B-Instruct --dir $DATA_DIR

## 5. View the text report
The report file is saved under `/kaggle/working/project/` — download it from the **Output** tab on the right.

In [ ]:
print(open('bench_report_transformers.txt', encoding='utf-8').read())